In [ ]:
import pyspark.sql.functions as F

In [ ]:
catalog = 'dev'
pipeline_id = dbutils.widgets.get('pipeline_id')
task_id = dbutils.widgets.get('task_id')
run_id = dbutils.widgets.get('run_id')
processed_time = dbutils.widgets.get('processed_time')

In [ ]:
base_path = (f"/Volumes/{catalog}/landing/raw_data/operational_data/")

In [ ]:
checkpoint_path = f"abfss://{catalog}@dablake.dfs.core.windows.net/bronze/spotify_events/_checkpoint/"

In [ ]:
schema_location = f"abfss://{catalog}@dablake.dfs.core.windows.net/bronze/spotify_events/_schema/"

In [ ]:
column_map = F.create_map(
    F.lit("pipeline_id"), F.lit(pipeline_id),
    F.lit("task_id"), F.lit(task_id),
    F.lit("run_id"), F.lit(run_id),
    F.lit("processed_time"), F.lit(processed_time)
)

In [ ]:
df = (spark.readStream.format("cloudFiles")
      .option('cloudfiles.format', 'json')
        .option('cloudfiles.inferColumnTypes', 'true')
        .option('cloudfiles.schemaLocation',schema_location)
        .load(f"{base_path}/spotify_events/")
        .withColumn("file_path",F.col("_metadata.file_path"))
        .withColumn('metadata', column_map)
        .writeStream
        .format("delta")
        .option("checkpointLocation", checkpoint_path)
        .trigger(availableNow=True)
        .toTable(f"{catalog}.bronze.spotify_events")
      )
df.awaitTermination()

In [ ]:
tracks_df =(spark.read.format("csv")
      .option('header', 'true')
      .load(f'{base_path}/spotify_dims/tracks/')
      .withColumn('metadata', column_map)
      .withColumn("file_path",F.col("_metadata.file_path")))
tracks_df.write.mode("overwrite").saveAsTable(f"{catalog}.bronze.spotify_tracks")


In [ ]:
users_df =  (spark.read.format("csv")
      .option('header', 'true')
      .load(f'{base_path}/spotify_dims/users/')
      .withColumn('metadata', column_map)
      .withColumn("file_path",F.col("_metadata.file_path")))
users_df.write.mode("overwrite").saveAsTable(f"{catalog}.bronze.spotify_users")